# Master Orchestration: Lazarus Training Mission
Immerse yourself in a guided control room that coordinates every model run, documents the artifacts, and keeps the laptop-friendly guardrails engaged.

## Mission Flow
1. Establish mission parameters and confirm the hardware envelope.
2. Load the global configuration and craft dataloaders sourced strictly from `./Data`.
3. Execute head-only warmups, optional fine-tunes, explainability exports, and experiment indexing.
4. Archive every artifact in the canonical structure for dashboard discovery.
5. Validate the pipeline with a fast smoke simulation before committing to longer voyages.

In [ ]:
# === Parameter Flight Deck ===
# selected_models = ["efficientnet_b0", "mobilenet_v3_small"]
# image_size_override = 224
# batch_size_override = 8
# num_epochs_override = 10
# freeze_backbone_default = True
# fast_test_mode = False
selected_models: list[str] | None = None
image_size_override: int | None = None
batch_size_override: int | None = None
num_epochs_override: int | None = None
freeze_backbone_default: bool | None = None
fast_test_mode: bool = False

In [ ]:
from __future__ import annotations
import json
import os
from pathlib import Path
import textwrap
import torch
import psutil
from datetime import datetime
from typing import Any

from src.master_trainer import MasterTrainer
from src.data_utils_torch import make_dataloaders

PROJECT_ROOT = Path.cwd().resolve()
DATA_ROOT = PROJECT_ROOT / "Data"
CONFIG_PATH = PROJECT_ROOT / "config.yaml"

In [ ]:
def describe_hardware() -> dict[str, Any]:
    snapshot: dict[str, Any] = {"torch_version": torch.__version__}
    snapshot["cuda_available"] = torch.cuda.is_available()
    if torch.cuda.is_available():
        snapshot["cuda_device"] = torch.cuda.get_device_name(0)
        props = torch.cuda.get_device_properties(0)
        snapshot["cuda_total_memory_gb"] = round(props.total_memory / 1e9, 2)
    virtual_mem = psutil.virtual_memory()
    snapshot["system_ram_gb"] = round(virtual_mem.total / 1e9, 2)
    snapshot["recommended_batch"] = 8 if virtual_mem.total >= 12 * 1024**3 else 4
    snapshot["recommended_image_size"] = 224
    snapshot["timestamp"] = datetime.utcnow().isoformat()
    return snapshot

hardware_report = describe_hardware()
if not DATA_ROOT.exists():
    raise FileNotFoundError("Data folder './Data' not found. Please place dataset in root/Data.")

print(textwrap.dedent(f"
Laptop mission profile:
  Torch version: {hardware_report['torch_version']}
  CUDA available: {hardware_report['cuda_available']}
  System RAM (GB): {hardware_report['system_ram_gb']}
  Recommended batch: {hardware_report['recommended_batch']}
  Recommended image size: {hardware_report['recommended_image_size']}
"))

In [ ]:
with CONFIG_PATH.open("r", encoding="utf-8") as handle:
    global_config = json.loads(json.dumps(__import__('yaml').safe_load(handle)))

suite_config = global_config.get("training_suite", {})
requested_models = selected_models or [entry.get('name') for entry in global_config.get('models', [])]
requested_models = [name for name in requested_models if name]

print('Resolved model itinerary:', requested_models)
print('Fast test mode:', fast_test_mode)

In [ ]:
trainer = MasterTrainer()
training_kwargs = {}
if batch_size_override is not None:
    training_kwargs['batch_size_override'] = batch_size_override
if image_size_override is not None:
    training_kwargs['image_size_override'] = image_size_override
if num_epochs_override is not None:
    training_kwargs['num_epochs_override'] = num_epochs_override
if freeze_backbone_default is not None:
    training_kwargs['freeze_backbone_default'] = freeze_backbone_default

mission_results = trainer.run(model_names=requested_models, fast_test=fast_test_mode)
print(f'Completed {len(mission_results)} model runs.')

In [ ]:
from pprint import pprint
pprint(mission_results)

In [ ]:
print('Initiating fast smoke validation on a classifier-head only subset.')
smoke_outcome = trainer.run(model_names=requested_models[:1] if requested_models else None, fast_test=True)
print('Smoke validation complete:')
pprint(smoke_outcome)